In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.metrics import classification_report, accuracy_score, f1_score

# Load artifacts
with open('medisense_preprocessed.pkl', 'rb') as f:
    data = pickle.load(f)

from xgboost import XGBClassifier
xgb = XGBClassifier()
xgb.load_model('xgb_model_lite.json')

le            = data['label_encoder']
feature_names = data['feature_names']
X_test        = data['X_test']
y_test        = data['y_test']

df = pd.read_csv('Final_Augmented_dataset_Diseases_and_Symptoms.csv')
df = df.rename(columns={'diseases': 'disease'})
symptom_cols = [c for c in df.columns if c != 'disease']

# ── CSV 1: Disease distribution ───────────────────────────
disease_counts = df['disease'].value_counts().reset_index()
disease_counts.columns = ['disease', 'record_count']
disease_counts.to_csv('tableau_01_disease_distribution.csv', index=False)
print(f"CSV 1 done: {len(disease_counts)} diseases")

# ── CSV 2: Symptom frequency ──────────────────────────────
symptom_freq = df[symptom_cols].sum().reset_index()
symptom_freq.columns = ['symptom', 'frequency']
symptom_freq = symptom_freq.sort_values('frequency', ascending=False)
symptom_freq.to_csv('tableau_02_symptom_frequency.csv', index=False)
print(f"CSV 2 done: {len(symptom_freq)} symptoms")

# ── CSV 3: Disease-symptom matrix (top 30 diseases) ───────
top_diseases  = disease_counts.head(30)['disease'].tolist()
df_top        = df[df['disease'].isin(top_diseases)]
disease_symptom = df_top.groupby('disease')[symptom_cols].mean().reset_index()
melted = disease_symptom.melt(
    id_vars='disease',
    var_name='symptom',
    value_name='avg_presence'
)
melted = melted[melted['avg_presence'] > 0.1]
melted.to_csv('tableau_03_disease_symptom_matrix.csv', index=False)
print(f"CSV 3 done: {len(melted)} rows")

# ── CSV 4: Model performance per disease ──────────────────
y_pred  = xgb.predict(X_test)
report  = classification_report(
    y_test, y_pred,
    target_names=le.classes_,
    output_dict=True,
    zero_division=0
)
perf_rows = []
for disease, metrics in report.items():
    if isinstance(metrics, dict) and disease in le.classes_:
        perf_rows.append({
            'disease'  : disease,
            'precision': round(metrics['precision'], 3),
            'recall'   : round(metrics['recall'], 3),
            'f1_score' : round(metrics['f1-score'], 3),
            'support'  : int(metrics['support'])
        })
df_perf = pd.DataFrame(perf_rows)
df_perf.to_csv('tableau_04_model_performance.csv', index=False)
print(f"CSV 4 done: {len(df_perf)} diseases")

# ── CSV 5: Overall model summary (KPI tiles) ──────────────
summary = pd.DataFrame([
    {'metric': 'Test Accuracy (%)',    'value': round(accuracy_score(y_test, y_pred)*100, 2)},
    {'metric': 'Weighted F1 (%)',      'value': round(f1_score(y_test, y_pred, average='weighted')*100, 2)},
    {'metric': 'Total Diseases',       'value': int(len(le.classes_))},
    {'metric': 'Total Symptoms',       'value': int(len(feature_names))},
    {'metric': 'Training Samples',     'value': 50000},
    {'metric': 'Test Samples',         'value': int(len(X_test))},
    {'metric': 'Total Dataset Rows',   'value': int(len(df))},
    {'metric': 'Knowledge Base Chunks','value': 1174},
])
summary.to_csv('tableau_05_model_summary.csv', index=False)
print(f"CSV 5 done")

# ── CSV 6: Top 20 diseases by F1 score (for bar race) ─────
top_f1 = df_perf.sort_values('f1_score', ascending=False).head(20)
top_f1.to_csv('tableau_06_top20_diseases_f1.csv', index=False)
print(f"CSV 6 done")

# ── CSV 7: Symptom co-occurrence across disease categories ─
# Group diseases into broad categories for Tableau treemap
category_map = {
    'respiratory': ['asthma', 'pneumonia', 'tuberculosis', 'bronchitis',
                    'chronic sinusitis', 'acute sinusitis', 'whooping cough'],
    'infectious' : ['malaria', 'dengue', 'typhoid', 'hepatitis', 'covid',
                    'chicken pox', 'measles', 'mumps'],
    'metabolic'  : ['diabetes', 'hypothyroidism', 'hyperthyroidism',
                    'obesity', 'hypoglycemia'],
    'cardiac'    : ['heart failure', 'hypertension', 'arrhythmia',
                    'coronary', 'angina'],
    'neurological': ['migraine', 'epilepsy', 'parkinson', 'alzheimer',
                     'meningitis', 'stroke'],
    'digestive'  : ['gastroenteritis', 'appendicitis', 'gerd', 'jaundice',
                    'hepatitis', 'pancreatitis', 'colitis'],
    'musculoskeletal': ['arthritis', 'osteoporosis', 'gout', 'fibromyalgia',
                        'spondylosis', 'sciatica'],
    'mental'     : ['anxiety', 'depression', 'panic disorder', 'insomnia',
                    'bipolar', 'schizophrenia'],
}

cat_rows = []
for disease, count in zip(disease_counts['disease'], disease_counts['record_count']):
    category = 'other'
    for cat, keywords in category_map.items():
        if any(kw in disease.lower() for kw in keywords):
            category = cat
            break
    cat_rows.append({
        'disease' : disease,
        'category': category,
        'record_count': count
    })

df_cat = pd.DataFrame(cat_rows)
df_cat.to_csv('tableau_07_disease_categories.csv', index=False)
print(f"CSV 7 done: {df_cat['category'].value_counts().to_dict()}")

print("\n✅ All 7 CSVs exported!")
print("Files ready for Tableau:")
for i in range(1, 8):
    import os
    fname = f'tableau_0{i}_*.csv'
    files = [f for f in os.listdir('.') if f.startswith(f'tableau_0{i}')]
    if files:
        size = os.path.getsize(files[0]) / 1024
        print(f"  {files[0]}: {size:.1f} KB")

/home/sunbeam/.local/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/sunbeam/.local/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/sunbeam/.local/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.8.0 when using version

CSV 1 done: 773 diseases
CSV 2 done: 377 symptoms
CSV 3 done: 360 rows
CSV 4 done: 677 diseases
CSV 5 done
CSV 6 done
CSV 7 done: {'other': 712, 'musculoskeletal': 11, 'cardiac': 9, 'digestive': 8, 'respiratory': 7, 'neurological': 7, 'mental': 7, 'metabolic': 6, 'infectious': 6}

✅ All 7 CSVs exported!
Files ready for Tableau:
  tableau_01_disease_distribution.csv: 17.1 KB
  tableau_02_symptom_frequency.csv: 8.2 KB
  tableau_03_disease_symptom_matrix.csv: 18.2 KB
  tableau_04_model_performance.csv: 25.0 KB
  tableau_05_model_summary.csv: 0.2 KB
  tableau_06_top20_diseases_f1.csv: 0.8 KB
  tableau_07_disease_categories.csv: 22.0 KB


In [1]:
# Better category mapping using partial string matching
import pandas as pd

df_cat = pd.read_csv('tableau_07_disease_categories.csv')

# More comprehensive keywords
category_map = {
    'Respiratory'    : ['asthma','pneumonia','tuberculosis','bronchitis','sinusitis',
                        'whooping','copd','emphysema','pleurisy','cough','lung'],
    'Infectious'     : ['malaria','dengue','typhoid','hepatitis','covid','chicken pox',
                        'measles','mumps','influenza','hiv','aids','infection','viral',
                        'bacterial','fungal','parasitic'],
    'Metabolic'      : ['diabetes','thyroid','obesity','hypoglycemia','gout','metabolic',
                        'cholesterol','lipid'],
    'Cardiac'        : ['heart','hypertension','arrhythmia','coronary','angina',
                        'cardiac','vascular','blood pressure','tachycardia'],
    'Neurological'   : ['migraine','epilepsy','parkinson','alzheimer','meningitis',
                        'stroke','nerve','neural','brain','seizure','tremor','vertigo'],
    'Digestive'      : ['gastro','appendicitis','gerd','jaundice','pancreatitis',
                        'colitis','bowel','liver','stomach','intestin','digestive',
                        'hepatitis','crohn','ulcer','reflux'],
    'Musculoskeletal': ['arthritis','osteo','gout','fibromyalgia','spondylosis',
                        'sciatica','joint','bone','muscle','spinal','back pain',
                        'tendon','ligament'],
    'Mental Health'  : ['anxiety','depression','panic','insomnia','bipolar',
                        'schizophrenia','mental','psychiatric','phobia','disorder',
                        'stress','ptsd','ocd','adhd'],
    'Skin'           : ['skin','dermat','eczema','psoriasis','acne','rash','allerg',
                        'hives','cellulitis','melanoma'],
    'Urological'     : ['kidney','urin','bladder','renal','prostate','cystitis',
                        'nephr','urinary'],
    'Reproductive'   : ['ovari','uterus','cervical','vaginal','menstrual','pregnancy',
                        'erectile','testicular','breast'],
    'Eye/ENT'        : ['eye','vision','ear','nose','throat','hearing','sinus',
                        'conjunctiv','glaucoma','cataract'],
}

def assign_category(disease):
    disease_lower = disease.lower()
    for cat, keywords in category_map.items():
        if any(kw in disease_lower for kw in keywords):
            return cat
    return 'Other'

df_cat['category'] = df_cat['disease'].apply(assign_category)
df_cat.to_csv('tableau_07_disease_categories.csv', index=False)

print(df_cat['category'].value_counts())

category
Other              487
Mental Health       53
Digestive           29
Infectious          26
Eye/ENT             26
Musculoskeletal     24
Urological          23
Cardiac             21
Reproductive        20
Skin                19
Metabolic           17
Neurological        15
Respiratory         13
Name: count, dtype: int64
